# 16 — Employee Intelligence Table

**Day 3, Step 16.** Everything from Days 2 and 3 lands here: one row per
employee with risk, engagement, role, skill gaps and the recommendation. The
dashboard is a view onto this table.

In [1]:
import sys, warnings
sys.path.insert(0, "../src"); sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# The lab imports from the factory. Nothing below reimplements pipeline logic.
from hrai.utils.config import get, raw_path, seed
from hrai.utils.io import load_raw, load_processed
from hrai.utils.logger import setup_logging
setup_logging(fmt="human")
print(f"seed={seed()}  |  datasets: {sorted(get('datasets'))}")

seed=42  |  datasets: ['employee_attrition', 'essential_skills', 'hr_performance_engagement', 'occupation_data', 'software_skills']


In [2]:
table = load_processed("employee_intelligence")
print(table.shape)
table.groupby(["population", "risk_band"]).size()

(4470, 27)


population  risk_band  
A           HIGH             58
            LOW            1228
            MEDIUM          184
B           UNAVAILABLE    3000
dtype: int64

## Population B carries no attrition probability — deliberately

The transfer validation (notebook 17) measured the restricted model at ROC-AUC
**0.50** on Population B: chance. Rather than print a plausible-looking number,
those rows carry `attrition_probability = null` and a stated reason.

A number indistinguishable from a coin toss is worse than an honest blank,
because someone will act on it.

In [3]:
cols = ["person_key", "department", "role", "attrition_probability", "risk_band",
        "engagement_score", "skill_gaps", "recommendation"]
print("POPULATION A — model-scored")
display(table[table.population == "A"].nlargest(5, "attrition_probability")[cols])
print("POPULATION B — risk withheld")
display(table[table.population == "B"].head(3)[cols])
print(table[table.population == "B"]["risk_unavailable_reason"].iloc[0])

POPULATION A — model-scored


,person_key,department,role,attrition_probability,risk_band,engagement_score,skill_gaps,recommendation
463,A-622,Research & Development,Laboratory Technician,0.9355,HIGH,<NA>,"Microsoft Outlook, Microsoft PowerPoint, Billi...",Technical Skills
911,A-1273,Sales,Sales Representative,0.9297,HIGH,<NA>,"Microsoft Outlook, Microsoft PowerPoint, Activ...",Technical Skills
1060,A-1494,Research & Development,Laboratory Technician,0.9155,HIGH,<NA>,"Microsoft Office, SAP, Billing software, Data ...",Business Writing and Documentation
357,A-478,Sales,Sales Representative,0.8905,HIGH,<NA>,"Adobe Creative Cloud software, Adobe InDesign,...",Cloud Platforms and Deployment
457,A-614,Sales,Sales Representative,0.8741,HIGH,<NA>,"Microsoft Office, Microsoft Outlook, Adobe Acr...",Business Writing and Documentation


POPULATION B — risk withheld


,person_key,department,role,attrition_probability,risk_band,engagement_score,skill_gaps,recommendation
1470,B-1001,Software Engineering,Software Engineer,<NA>,UNAVAILABLE,2,"Hypertext markup language HTML, IBM Terraform,...",Technical Skills
1471,B-1002,Software Engineering,Software Engineer,<NA>,UNAVAILABLE,4,"Atlassian JIRA, Cascading style sheets CSS, Do...",Containerisation and Deployment Pipelines
1472,B-1003,Software Engineering,Software Engineer,<NA>,UNAVAILABLE,2,"Amazon Web Services AWS software, Apache Kafka...",Cloud Platforms and Deployment


The attrition model does not transfer to this population (externally validated ROC-AUC 0.5039 against its own outcomes — indistinguishable from chance). Reporting a risk score here would be misleading, so none is shown.
